# Single Prompt Evaluation with Tree of Thoughts (ToT)

This notebook evaluates the `gemini-3.1-flash-lite` model on Killer Sudoku puzzles 1-5 (6x6) using a Single Prompt format enhanced with Tree of Thoughts (ToT) self-correction and sampling search.

In [11]:
import os
import json
import time
import sys
from pathlib import Path
from google import genai
from google.genai import types

# Setup Google GenAI Client
os.environ["GOOGLE_CLOUD_PROJECT"] = "project-038ccd57-3d62-4aac-8b5"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

client = genai.Client()
model_LLM = "gemini-3.1-flash-lite"

In [12]:
# Dynamic path resolution to ensure it runs from any context
cwd = Path.cwd()
if (cwd / 'tot_solver.py').exists():
    tot_dir = cwd
elif (cwd / 'ToT' / 'tot_solver.py').exists():
    tot_dir = cwd / 'ToT'
elif (cwd / 'cs106' / 'ToT' / 'tot_solver.py').exists():
    tot_dir = cwd / 'cs106' / 'ToT'
else:
    raise FileNotFoundError('Cannot locate cs106/ToT resources from current directory.')

sys.path.insert(0, str(tot_dir.resolve()))
from tot_solver import run_single_prompt_tot_search

cs106_dir = tot_dir.parent
dataset_dir = cs106_dir / 'dataset'
output_dir = tot_dir / 'outputs' / 'single-prompt'
output_dir.mkdir(parents=True, exist_ok=True)

print(f"ToT directory: {tot_dir}")
print(f"Dataset directory: {dataset_dir}")
print(f"Output directory: {output_dir}")

ToT directory: d:\Study\HK6\CS106-AI\DoAn\CS106-Sudoku-Bench\cs106\ToT
Dataset directory: d:\Study\HK6\CS106-AI\DoAn\CS106-Sudoku-Bench\cs106\dataset
Output directory: d:\Study\HK6\CS106-AI\DoAn\CS106-Sudoku-Bench\cs106\ToT\outputs\single-prompt


In [13]:
prompt_6x6 = """
You are an expert puzzle solver. Your task is to solve a 6x6 Killer Sudoku grid based on the rules, strategies, and puzzle data provided below.

1. The Grid & Core Rules:
- The grid is 6x6, containing 6 rows and 6 columns.
- Notation: "r1c1" indicates the cell at row 1 and column 1.
- The grid consists of 6 rectangular 2x3 subgrids (2 rows tall, 3 columns wide).
- Standard Sudoku Rule: Each row, each column, and each 2x3 subgrid must contain the numbers 1 through 6 exactly once.
- Killer Sudoku Rule: The grid is divided into "Cages" (contiguous groups of cells). The sum of the numbers in each cage must perfectly match its provided target sum.
- Non-Repeating Rule: Numbers cannot repeat within a single cage.

2. Mathematical Facts & Solving Strategies:
- Sum to 21: Since the numbers 1 through 6 add up to 21, every completely filled row, column, and 2x3 subgrid must sum exactly to 21.
- A 6-cell cage must also have a target sum of 21.
- Deduction Tip: For any subset of cells (row, column, or subgrid), if all but one cell are filled, the final cell's value must be 21 minus the sum of the known cells.

3. Reference Material: The Cheat-Sheet
Important Interpretation Rules for the cheat-sheet:
- Combinations are written as sequences of individual digits without separators.
- Each digit in the sequence represents one distinct number in the combination.
- Example: "12" means the combination {{1, 2}}. "135" means the combination {{1, 3, 5}}.
Here is your cheat-sheet for cage combinations:
{cheat_sheet}

4. The Puzzle Instance:
Initial State:
{puzzle}

Cages:
{cages}

5. Output Requirement:
Return ONLY the final solved 6x6 grid. Do not include any explanation or extra text.
"""

prompt_9x9 = """
You are an expert Killer Sudoku solver.

Your task is to solve the Killer Sudoku puzzle based on the rules, reference material, and puzzle data provided below.

1. Grid & Core Rules

- The grid size is {grid_size}x{grid_size}.
- Notation: "r1c1" means row 1, column 1.
- Each cell must contain one digit from 1 to {grid_size}.
- Subgrid shape: {box_shape}.

Standard Sudoku Rules:
- Each row must contain digits 1 to {grid_size} exactly once.
- Each column must contain digits 1 to {grid_size} exactly once.
- Each subgrid must contain digits 1 to {grid_size} exactly once.

Killer Sudoku Rules:
- The grid is divided into cages.
- The digits in each cage must sum to the cage target.
- Digits cannot repeat within the same cage.

2. Mathematical Facts

For this puzzle:
- Row sum: {unit_sum}
- Column sum: {unit_sum}
- Subgrid sum: {unit_sum}
- Number of cage equations: {cage_count}
- Total sum equations: {total_equations}

3. Reference Material: Killer Sudoku Cheat Sheet

- Combinations are written as digit sequences without separators.
- Example: "12" means {{1, 2}}
- Example: "135" means {{1, 3, 5}}

Cheat sheet:

{cheat_sheet}

4. Puzzle Instance

Initial State:

{puzzle}

Cages:

{cages}

5. Solving Requirements

Solve the puzzle logically using:
- Sudoku row constraints
- Sudoku column constraints
- Sudoku subgrid constraints
- Killer cage sum constraints
- Cage distinctness constraints
- Cheat sheet combinations

Do NOT guess.
Do NOT invent values.
Only return a solution if it is logically consistent with ALL constraints.

6. Mandatory Self-Verification Before Output

Before returning the final solution, silently verify ALL of the following:

Sudoku validation:
- Every row contains digits 1 to {grid_size} exactly once.
- Every column contains digits 1 to {grid_size} exactly once.
- Every subgrid contains digits 1 to {grid_size} exactly once.

Killer Sudoku validation:
- Every cage sums exactly to its target.
- No repeated digit exists inside any cage.

Global validation:
- Every cell is filled.
- No contradictions exist.

If ANY validation fails:
DO NOT return an invalid board.

7. Output Requirement

Return ONLY valid JSON matching this schema:

{{
  "board": [[...]]
}}

If you cannot determine a fully valid solution with certainty, return:

{{
  "board": []
}}

Do not include:
- explanations
- markdown
- comments
- reasoning text
- extra text
"""

In [14]:
puzzle_ids = list(range(7, 9))
all_results = []

for pid in puzzle_ids:
    p_file = dataset_dir / f"puzzle_{pid:02d}.json"
    print(f"\nEvaluating Puzzle {pid:02d} ({p_file.name})...")
    
    with open(p_file, 'r', encoding='utf-8') as f:
        puzzle = json.load(f)
        
    grid_size = puzzle['grid_size']
    
    # Select prompt template and cheat sheet
    if grid_size == 6:
        prompt_template = prompt_6x6
        cheat_sheet_file = cs106_dir / "6x6" / "killer_sudoku_cheat_sheet.md"
    else:
        prompt_template = prompt_9x9
        cheat_sheet_file = cs106_dir / "9x9" / "killer_sudoku_cheat_sheet_9x9.md"
        
    with open(cheat_sheet_file, 'r', encoding='utf-8') as f:
        cheat_sheet = f.read()

    # Form cages text
    cages_text = []
    for cage in puzzle['cages']:
        cell_strs = [f"r{r+1}c{c+1}" for r, c in cage['cells']]
        cages_text.append("- " + " + ".join(cell_strs) + f" = {cage['sum']}")
    
    start_time = time.time()
    
    # Run single-prompt ToT
    solved_board, logs, status = run_single_prompt_tot_search(
        client=client,
        model=model_LLM,
        puzzle=puzzle,
        prompt_template=prompt_template,
        cages_text=cages_text,
        cheat_sheet=cheat_sheet,
        max_corrections=5,
        k_samples=3,
        temperature=0.7
    )
    
    elapsed = time.time() - start_time
    
    log_data = {
        "puzzle_id": puzzle['id'],
        "difficulty": puzzle['difficulty'],
        "grid_size": grid_size,
        "model": model_LLM,
        "status": status,
        "time_seconds": elapsed,
        "prediction": solved_board,
        "solution": puzzle['solution'],
        "tot_logs": logs
    }
    
    out_file = output_dir / f"log_puzzle_{pid:02d}.json"
    with open(out_file, 'w', encoding='utf-8') as f:
        json.dump(log_data, f, ensure_ascii=False, indent=2)
        
    print(f"  Result: {status} | Time: {elapsed:.2f}s | Saved to: {out_file.name}")
    all_results.append(log_data)
    
    time.sleep(2) # respect rate limits


Evaluating Puzzle 07 (puzzle_07.json)...
  Result: Success | Time: 214.07s | Saved to: log_puzzle_07.json

Evaluating Puzzle 08 (puzzle_08.json)...


FileNotFoundError: [Errno 2] No such file or directory: 'd:\\Study\\HK6\\CS106-AI\\DoAn\\CS106-Sudoku-Bench\\cs106\\dataset\\puzzle_08.json'

## 3. Summary of runs

In [ ]:
print("\nEvaluation Complete!")
print("-" * 40)
for r in all_results:
    print(f"Puzzle {r['puzzle_id']:02d} ({r['grid_size']}x{r['grid_size']}) | {r['status']} | Time: {r['time_seconds']:.2f}s")


Evaluation Complete!
----------------------------------------
Puzzle 06 (9x9) | Success | Time: 162.86s
